In [1]:
!pip install ultralytics

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.2/42.2 kB 2.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 61.5 MB/s eta 0:00:00


## Stage 1: YOLO-Based Nodule Detection and Region of Interest (ROI) Extraction

In [2]:
from ultralytics import YOLO
import cv2

class YOLODetector:

    def __init__(self, model_path, device=0):
        """
        Load YOLO model once.
        """
        self.model = YOLO(model_path)
        self.device = device

    def detect_nodule(self, image_path):

        results = self.model.predict(
            source=image_path,
            conf=0.25,
            device=self.device,
            verbose=False
        )

        result = results[0]

        # No detection
        if len(result.boxes) == 0:
            return {
                "nodule_detected": False,
                "bounding_box": None,
                "detection_confidence": 0.0,
                "roi": None
            }

        # Highest confidence box
        box = result.boxes[0]

        xmin, ymin, xmax, ymax = box.xyxy[0].cpu().numpy().astype(int)

        confidence = float(box.conf[0])

        # Read original image
        image = cv2.imread(image_path)

        if image is None:
            raise FileNotFoundError(f"Could not read {image_path}")

        roi = image[ymin:ymax, xmin:xmax]
        return {
            "nodule_detected": True,
            "bounding_box": {
                "xmin": int(xmin),
                "ymin": int(ymin),
                "xmax": int(xmax),
                "ymax": int(ymax)
            },
            "detection_confidence": float(confidence),
            "roi": roi
        }

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.


## Load the Trained YOLO11m Nodule Detection Model

In [3]:
import torch
device = 0 if torch.cuda.is_available() else "cpu"
detector = YOLODetector(
    model_path="/kaggle/input/datasets/anikasadiaopt/dataset-thyroid/results_YOLO/YOLO_for_detection/YOLO_Thyroid_Nodule_Detection/YOLO11m/weights/best.pt",
    

    device = device
)

## Run YOLO Nodule Detection on the Test Dataset

In [4]:
from pathlib import Path

test_dir = Path("/kaggle/input/datasets/anikasadiaopt/test-set/test")

image_paths = sorted(test_dir.glob("*.jpg"))

print(f"Total test images: {len(image_paths)}")
detected = 0
missed = 0
for img_path in image_paths:

    result = detector.detect_nodule(str(img_path))

    if result["nodule_detected"]:
        detected += 1
    else:
        missed += 1
        
    print(img_path.name)
    print(result)
    print("-" * 50)
print(f"Detected: {detected}")
print(f"Missed: {missed}")

Total test images: 500
000009.jpg
{'nodule_detected': True, 'bounding_box': {'xmin': 217, 'ymin': 84, 'xmax': 627, 'ymax': 337}, 'detection_confidence': 0.8382206559181213, 'roi': array([[[57, 57, 57],
        [57, 57, 57],
        [56, 56, 56],
        ...,
        [81, 81, 81],
        [82, 82, 82],
        [83, 83, 83]],

       [[56, 56, 56],
        [56, 56, 56],
        [56, 56, 56],
        ...,
        [85, 85, 85],
        [86, 86, 86],
        [88, 88, 88]],

       [[52, 52, 52],
        [53, 53, 53],
        [55, 55, 55],
        ...,
        [90, 90, 90],
        [91, 91, 91],
        [93, 93, 93]],

       ...,

       [[23, 23, 23],
        [23, 23, 23],
        [23, 23, 23],
        ...,
        [ 5,  5,  5],
        [ 4,  4,  4],
        [ 2,  2,  2]],

       [[21, 21, 21],
        [20, 20, 20],
        [19, 19, 19],
        ...,
        [ 6,  6,  6],
        [ 4,  4,  4],
        [ 1,  1,  1]],

       [[29, 29, 29],
        [24, 24, 24],
        [20, 20, 20],
      

## Stage-2: Loaded and Prepared the CNN Model for Thyroid Nodule Classification


In [5]:
import cv2
import torch
import torch.nn as nn
import albumentations as A
from albumentations.pytorch import ToTensorV2


# --------------------------------------------------
# Device
# --------------------------------------------------
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

class ThyroidCNN(nn.Module):
    def __init__(self):
        super().__init__()

        self.features = nn.Sequential(

            nn.Conv2d(3, 32, 3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(32, 64, 3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(64, 128, 3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(128, 256, 3, padding=1),
            nn.BatchNorm2d(256),
            nn.ReLU(),
            nn.AdaptiveAvgPool2d(1)
        )

        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Dropout(0.5),
            nn.Linear(256, 128),
            nn.ReLU(),
            nn.Dropout(0.5),
            nn.Linear(128, 2))
    def forward(self, x):
        x = self.features(x)
        x = self.classifier(x)
        return x
model = ThyroidCNN()

model.load_state_dict(
    torch.load(
        "/kaggle/input/datasets/anikasadiaopt/dataset-thyroid/results/Output/model/Classification/thyroid_cnn.pt",        # <-- Change this path
        map_location=device
    )
)
model.to(device)
model.eval()
transform = A.Compose([
    A.Resize(224, 224),
    A.Normalize(
        mean=(0.485, 0.456, 0.406),
        std=(0.229, 0.224, 0.225)
    ),
    ToTensorV2()
])

class_names = {
    0: "Benign",
    1: "Malignant"
}

roi_output_dir = Path("results/roi")
roi_output_dir.mkdir(parents=True, exist_ok=True)
def predict(roi, image_name):

    roi_path = roi_output_dir / f"{image_name}_roi.png"

    cv2.imwrite(str(roi_path), roi)

    img = cv2.cvtColor(roi, cv2.COLOR_BGR2RGB)

    image = transform(image=img)["image"]
    image = image.unsqueeze(0).to(device)

    with torch.no_grad():

        outputs = model(image)

        probs = torch.softmax(outputs, dim=1)

        confidence, pred = torch.max(probs, dim=1)

    result = {
        "class_id": pred.item(),
        "classification": class_names[pred.item()],
        "confidence": float(confidence.item()),
        "roi_path": str(roi_path)
    }

    return result

## Stage-3: Integrated YOLO Detection and CNN Classification Pipeline on Test Images


In [6]:
from pathlib import Path

test_dir = Path("/kaggle/input/datasets/anikasadiaopt/test-set/test")
image_paths = sorted(test_dir.glob("*.jpg"))

results = []

for img_path in image_paths:

    # ---------------- YOLO ----------------
    yolo_result = detector.detect_nodule(str(img_path))

    if not yolo_result["nodule_detected"]:
        print(f"{img_path.name}: No nodule detected.")
        continue

    roi = yolo_result["roi"]

    # ---------------- CNN ----------------
    cnn_result = predict(roi, img_path.stem)

    # ---------------- Store results ----------------
    result = {
        "patient_id": img_path.stem,
        "image_name": img_path.name,
        "nodule_detected": True,
        "bounding_box": yolo_result["bounding_box"],
        "detection_confidence": yolo_result["detection_confidence"],
        "classification": cnn_result["classification"],
        "classification_confidence": cnn_result["confidence"]
    }

    results.append(result)

    print(result)

{'patient_id': '000009', 'image_name': '000009.jpg', 'nodule_detected': True, 'bounding_box': {'xmin': 217, 'ymin': 84, 'xmax': 627, 'ymax': 337}, 'detection_confidence': 0.8382206559181213, 'classification': 'Malignant', 'classification_confidence': 0.6823826432228088}
{'patient_id': '000018', 'image_name': '000018.jpg', 'nodule_detected': True, 'bounding_box': {'xmin': 135, 'ymin': 64, 'xmax': 643, 'ymax': 482}, 'detection_confidence': 0.9341974258422852, 'classification': 'Benign', 'classification_confidence': 0.8363583087921143}
{'patient_id': '000030', 'image_name': '000030.jpg', 'nodule_detected': True, 'bounding_box': {'xmin': 94, 'ymin': 110, 'xmax': 376, 'ymax': 344}, 'detection_confidence': 0.9059850573539734, 'classification': 'Benign', 'classification_confidence': 0.8139492273330688}
{'patient_id': '000044', 'image_name': '000044.jpg', 'nodule_detected': True, 'bounding_box': {'xmin': 404, 'ymin': 76, 'xmax': 461, 'ymax': 129}, 'detection_confidence': 0.7865909934043884, 'c

In [7]:
!pip install grad-cam

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.8/7.8 MB 99.6 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Created wheel for grad-cam: filename=grad_cam-1.5.5-py3-none-any.whl size=44285 sha256=b70f02c39514fae0020ffa68cbc616f9aaf505b412b27527af6e9cf080771566
  Stored in directory: /root/.cache/pip/wheels/fb/3b/09/2afc520f3d69bc26ae6bd87416759c820a3f7d05c1a077bbf6
Successfully built grad-cam


## Stage - 4: Implement Grad-CAM for CNN-Based Thyroid Nodule Explainability


In [8]:
import cv2
import torch
import numpy as np

from pytorch_grad_cam import GradCAM
from pytorch_grad_cam.utils.image import show_cam_on_image


class GradCAMGenerator:

    def __init__(self, model, transform, device):

        self.model = model
        self.device = device
        self.transform = transform

        self.model.eval()

        # Last Conv2d layer of your ThyroidCNN
        self.target_layers = [self.model.features[12]]

        self.cam = GradCAM(
            model=self.model,
            target_layers=self.target_layers
        )

    def generate(self, roi, save_path):

        # roi is already a BGR NumPy array from YOLO
    
        img = cv2.cvtColor(roi, cv2.COLOR_BGR2RGB)
        img = cv2.resize(img, (224, 224))
    
        rgb_img = img.astype(np.float32) / 255.0
    
        input_tensor = self.transform(
            image=img
        )["image"].unsqueeze(0).to(self.device)
    
        outputs = self.model(input_tensor)
    
        probabilities = torch.softmax(outputs, dim=1)
    
        pred = probabilities.argmax(dim=1).item()
    
        confidence = probabilities[0, pred].item()
    
        grayscale_cam = self.cam(input_tensor=input_tensor)[0]

        visualization = show_cam_on_image(
            rgb_img,
            grayscale_cam,
            use_rgb=True
        )
    
        save_path = Path(save_path)
        save_path.parent.mkdir(parents=True, exist_ok=True)
    
        cv2.imwrite(
            str(save_path),
            cv2.cvtColor(visualization, cv2.COLOR_RGB2BGR)
        )
    
        return {
            "prediction": pred,
            "confidence": confidence,
            "gradcam_path": str(save_path)
        }

In [9]:
from pathlib import Path

# Initialize Grad-CAM
gradcam = GradCAMGenerator(
    model=model,
    transform=transform,
    device=device
)

save_dir = Path("/kaggle/working/results/gradcam")
save_dir.mkdir(parents=True, exist_ok=True)

all_results = []

for img_path in image_paths:

    # ---------------- YOLO ----------------
    yolo_result = detector.detect_nodule(str(img_path))

    if not yolo_result["nodule_detected"]:
        continue

    bbox = yolo_result["bounding_box"]
    det_conf = yolo_result["detection_confidence"]
    roi = yolo_result["roi"]

    cnn_result = predict(roi, img_path.stem)

    classification = cnn_result["classification"]
    cls_conf = cnn_result["confidence"] 

    # ---------------- Grad-CAM ----------------
    save_path = save_dir / f"{img_path.stem}_gradcam.png"

    gradcam_result = gradcam.generate(
        roi=roi,                  # <-- ROI array from YOLO
        save_path=save_path
    )

    gradcam_result["patient_id"] = img_path.stem
    gradcam_result["image_name"] = img_path.name

    all_results.append(gradcam_result)

    print(gradcam_result)

print(f"\nProcessed {len(all_results)} images.")

{'prediction': 1, 'confidence': 0.6823826432228088, 'gradcam_path': '/kaggle/working/results/gradcam/000009_gradcam.png', 'patient_id': '000009', 'image_name': '000009.jpg'}
{'prediction': 0, 'confidence': 0.8363583087921143, 'gradcam_path': '/kaggle/working/results/gradcam/000018_gradcam.png', 'patient_id': '000018', 'image_name': '000018.jpg'}
{'prediction': 0, 'confidence': 0.8139492273330688, 'gradcam_path': '/kaggle/working/results/gradcam/000030_gradcam.png', 'patient_id': '000030', 'image_name': '000030.jpg'}
{'prediction': 0, 'confidence': 0.8211263418197632, 'gradcam_path': '/kaggle/working/results/gradcam/000044_gradcam.png', 'patient_id': '000044', 'image_name': '000044.jpg'}
{'prediction': 1, 'confidence': 0.7743858695030212, 'gradcam_path': '/kaggle/working/results/gradcam/000046_gradcam.png', 'patient_id': '000046', 'image_name': '000046.jpg'}
{'prediction': 0, 'confidence': 0.6020223498344421, 'gradcam_path': '/kaggle/working/results/gradcam/000062_gradcam.png', 'patient

In [10]:
import json
from pathlib import Path

def save_json(
    patient_id,
    image_name,
    nodule_detected,
    bounding_box,
    detection_confidence,
    classification,
    classification_confidence,
    gradcam_path
):

    # Convert NumPy integers to Python integers
    bounding_box = {
        "xmin": int(bounding_box["xmin"]),
        "ymin": int(bounding_box["ymin"]),
        "xmax": int(bounding_box["xmax"]),
        "ymax": int(bounding_box["ymax"])
    }

    output = {
        "patient_id": str(patient_id),
        "image_name": str(image_name),
        "nodule_detected": bool(nodule_detected),
        "bounding_box": bounding_box,
        "detection_confidence": round(float(detection_confidence), 4),
        "classification": str(classification),
        "classification_confidence": round(float(classification_confidence), 4),
        "gradcam_path": str(gradcam_path)
    }

    save_dir = Path("/kaggle/working/results/json")
    save_dir.mkdir(parents=True, exist_ok=True)

    json_file = save_dir / f"{patient_id}.json"

    with open(json_file, "w") as f:
        json.dump(output, f, indent=4)

    return json_file

In [11]:
from pathlib import Path 
for img_path in image_paths:

    # ---------------- YOLO ----------------
    yolo_result = detector.detect_nodule(str(img_path))

    # Skip if nothing detected
    if not yolo_result["nodule_detected"]:
        continue

    bbox = yolo_result["bounding_box"]
    det_conf = yolo_result["detection_confidence"]
    roi = yolo_result["roi"]

    cnn_result = predict(roi,img_path.stem)

    classification = cnn_result["classification"]
    cls_conf = cnn_result["confidence"]                # float

    save_path = Path("/kaggle/working/results/gradcam") / f"{img_path.stem}_gradcam.png"
    gradcam_result = gradcam.generate(
        roi=roi,
        save_path=save_path
    )
    gradcam_path=gradcam_result["gradcam_path"]
    # ---------------- Save JSON ----------------
    save_json(
        patient_id=img_path.stem,
        image_name=img_path.name,
        nodule_detected=True,
        bounding_box=bbox,
        detection_confidence=det_conf,
        classification=classification,
        classification_confidence=cls_conf,
        gradcam_path=str(gradcam_path)
    )

